# Patricia — mT5 Fine-Tuned for Somali

## Project Context
This notebook is my contribution to our group's NLP translation
project. The group is building machine translation models from English into
several low-resource languages spoken in Kenya (Dholuo, Somali, Ekegusii),
using two different base models: **mT5** and **NLLB**.

My assignment: fine-tune **mT5** to translate English → **Somali**.

Somali is natively supported by mT5's pretrained vocabulary, so
my task is fine-tuning no vocabulary or embedding
modifications required.

## Data Pipeline
The dataset originates from our group's shared public-service-announcement
(PSA) corpus, spanning English, Kiswahili, Dholuo, and Somali. Since Somali
coverage in the original combined dataset was only partial (~5,100 of
~16,000 rows had real Somali translations), the missing rows were filled in
using machine translation (Google Translate) before this notebook runs.

## What This Notebook Does
1. Loads the English–Somali dataset and splits it into train/validation/test
2. Loads a pretrained `mt5-small` model and tokenizer
3. Tokenizes the data for sequence-to-sequence training
4. Fine-tunes the model for 10 epochs, evaluating BLEU, sacreBLEU, and chrF
   after each epoch
5. Saves the best-performing checkpoint and generates sample translations
   for the final report

## Step 1: Mount Google Drive
Connects this Colab session to my Google Drive so that datasets, checkpoints,
and the final model are saved persistently which protects against progress
loss if the Colab runtime disconnects.

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Step 2: Set Up Project Folders
Defines my project's working directory in Drive and creates the subfolders
needed to organize data, training checkpoints, and the final saved model.

In [3]:
PROJECT_DIR = '/content/drive/MyDrive/NLP_Group_Project/Patricia_mT5_Somali'

import os
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/final_model', exist_ok=True)
print("Drive mounted. Project dir:", PROJECT_DIR)

Drive mounted. Project dir: /content/drive/MyDrive/NLP_Group_Project/Patricia_mT5_Somali


## Step 3: Download the Base Dataset
Pulls the group's combined English–Kiswahili–Dholuo–Somali dataset directly
from our shared GitHub repo into my Drive project folder. This file has
Somali filled in for only part of the rows (~5,100 of ~16,000) — the rest
get filled via machine translation in the next step.

In [ ]:
!wget https://raw.githubusercontent.com/SelmahT/psa-dholuo-mt/main/data/processed/psa_dataset_dholuo_somali.csv -O "{PROJECT_DIR}/data/psa_dataset_dholuo_somali.csv"

--2026-08-03 16:56:18--  https://raw.githubusercontent.com/SelmahT/psa-dholuo-mt/main/data/processed/psa_dataset_dholuo_somali.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12723842 (12M) [text/plain]
Saving to: ‘/content/drive/MyDrive/NLP_Group_Project/Patricia_mT5_Somali/data/psa_dataset_dholuo_somali.csv’

/content/drive/MyDr 100%[===================>]  12.13M  61.8MB/s    in 0.2s    

2026-08-03 16:56:19 (61.8 MB/s) - ‘/content/drive/MyDrive/NLP_Group_Project/Patricia_mT5_Somali/data/psa_dataset_dholuo_somali.csv’ saved [12723842/12723842]



## Step 4: Fill Missing Somali Translations via Machine Translation
Fills in the ~10,900 rows that were missing a Somali translation, using the
free Google Translate endpoint on the English source text. Each row is
tagged as "human" (original) or "machine (Google Translate)" so the split
between real and MT-generated targets is traceable later. Progress is saved
every 50 rows so the process can resume from where it left off if
interrupted, rather than starting over.

In [ ]:
import pandas as pd
import requests
import time
import os

INPUT_FILE = f'{PROJECT_DIR}/data/psa_dataset_dholuo_somali.csv'
OUTPUT_FILE = f'{PROJECT_DIR}/data/psa_dataset_5lang_with_mt_somali.csv'
CHECKPOINT_EVERY = 50  # save progress every 50 rows, in case of a crash/timeout

def translate_somali(text, source='en'):
    url = "https://translate.googleapis.com/translate_a/single"
    params = {'client': 'gtx', 'sl': source, 'tl': 'so', 'dt': 't', 'q': text}
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return ''.join([seg[0] for seg in r.json()[0]])

# Resume from a checkpoint if one already exists (in case this got interrupted)
if os.path.exists(OUTPUT_FILE):
    df = pd.read_csv(OUTPUT_FILE)
    print("Resuming from existing checkpoint file.")
else:
    df = pd.read_csv(INPUT_FILE)
    df["Somali_source"] = df["Somali"].apply(lambda x: "human" if pd.notna(x) else None)
    print("Starting fresh.")

missing_mask = df['Somali'].isna()
missing_idx = df[missing_mask].index.tolist()
print(f"Rows needing Somali translation: {len(missing_idx)}")

fail_count = 0
for n, i in enumerate(missing_idx):
    eng = df.at[i, 'English']
    try:
        df.at[i, 'Somali'] = translate_somali(eng)
        df.at[i, 'Somali_source'] = 'machine (Google Translate)'
    except Exception as e:
        fail_count += 1
        print(f"Failed at row {i}: {e}")
    time.sleep(0.4)  # be polite to the free endpoint, avoid getting rate-limited
    if (n + 1) % CHECKPOINT_EVERY == 0:
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"Checkpoint saved: {n + 1}/{len(missing_idx)} done")

df.to_csv(OUTPUT_FILE, index=False)
print(f"\nDone. Total failures: {fail_count}")
print("Remaining missing Somali:", df['Somali'].isna().sum())

Resuming from existing checkpoint file.
Rows needing Somali translation: 0

Done. Total failures: 0
Remaining missing Somali: 0


## Step 5: Reconnect After Runtime Reset
Re-mounts Drive and recreates the project folder variables. Needed because
Colab wipes all variables in memory whenever the runtime disconnects or
restarts — this cell restores the session state before continuing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/NLP_Group_Project/Patricia_mT5_Somali'

import os
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/final_model', exist_ok=True)
print("Drive mounted. Project dir:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Project dir: /content/drive/MyDrive/NLP_Group_Project/Patricia_mT5_Somali


## Step 6: Install Required Libraries
Installs the Python packages needed for model training and evaluation:
`transformers` and `datasets` (model + data handling), `evaluate`,
`sacrebleu`, and `scikit-learn` (metrics and train/test splitting),
`sentencepiece` (mT5's tokenizer), and `accelerate` (training performance).

In [5]:
!pip install -q transformers datasets evaluate sacrebleu sentencepiece accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 6.8 MB/s eta 0:00:00


## Step 7: Import Libraries and Check GPU
Imports everything needed for training (data handling, PyTorch, Hugging Face
Transformers/Datasets/Evaluate) and confirms a GPU is attached to this
runtime since fine-tuning a transformer model on CPU alone would be
impractically slow.

In [6]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
import evaluate

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


## Step 8: Load and Split the Dataset
Loads the completed English–Somali dataset (with all missing translations
now filled), keeps only the source/target columns needed for training, and
splits it into train (80%), validation (10%), and test (10%) sets using a
fixed random seed for reproducibility. Each split is saved to Drive.

In [7]:
DATA_PATH = f'{PROJECT_DIR}/data/psa_dataset_5lang_with_mt_somali.csv'
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['English', 'Somali']).reset_index(drop=True)
df = df.rename(columns={'English': 'source', 'Somali': 'target'})[['source', 'target']]

from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

Train: 12823 Val: 1603 Test: 1603


## Step 9: Load Pretrained mT5 Model and Tokenizer
Loads the pretrained `mt5-small` model and its tokenizer, and moves the
model onto the GPU. Since Somali is already part of mT5's pretrained
vocabulary, no tokenizer modifications or embedding resizing are needed hence
this is straightforward fine-tuning.

In [ ]:
MODEL_NAME = "google/mt5-small"  # Somali is natively supported — no new tokens/embeddings needed

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to("cuda")

print(f"Loaded {MODEL_NAME} on GPU — vocab size: {tokenizer.vocab_size}")

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.20GB            

...

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loaded google/mt5-small on GPU — vocab size: 250100


In [8]:
MODEL_NAME = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(f'{PROJECT_DIR}/checkpoints/checkpoint-19236')
model = model.to("cuda")

nan_params = sum(1 for _, p in model.named_parameters() if torch.isnan(p).any() or torch.isinf(p).any())
print(f"Corrupted parameters: {nan_params} (should be 0)")

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Corrupted parameters: 0 (should be 0)


## Step 10: Tokenize the Dataset
Converts the English source text and Somali target text into token IDs the
model can process, capping each at 128 tokens. Padding tokens in the labels
are replaced with -100 so the model doesn't get penalized for predicting
padding during training. A data collator is set up to batch examples
together efficiently at training time.

In [9]:
MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 128

def preprocess(batch):
    inputs = [str(x) for x in batch["source"]]
    targets = [str(x) for x in batch["target"]]

    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LEN, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=MAX_TARGET_LEN, truncation=True, padding="max_length")
    labels["input_ids"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})

tokenized_datasets = raw_datasets.map(preprocess, batched=True, remove_columns=raw_datasets["train"].column_names)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
print("Tokenization complete.")

Map:   0%|          | 0/12823 [00:00<?, ? examples/s]

Map:   0%|          | 0/1603 [00:00<?, ? examples/s]

Map:   0%|          | 0/1603 [00:00<?, ? examples/s]

Tokenization complete.


## Diagnostic: Quick Training Test on a Small Sample
Debugging cell used to quickly test whether the model trains correctly on
just 50 rows, without waiting through a full epoch on the whole dataset.
This was used to catch two real bugs before committing to a full training
run: first, that fp16 mixed-precision was producing NaN losses, and later,
that the model's weights had become corrupted from an earlier failed run
and needed to be reloaded fresh. Not part of the main training pipeline.

In [ ]:
small_train = tokenized_datasets["train"].select(range(50))

diag_args = Seq2SeqTrainingArguments(
    output_dir=f'{PROJECT_DIR}/diag_checkpoints',
    eval_strategy="no",
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    learning_rate=3e-4,
    logging_strategy="steps",
    logging_steps=1,
    predict_with_generate=False,
    fp16=False,
    report_to="none",
)

diag_trainer = Seq2SeqTrainer(
    model=model,
    args=diag_args,
    train_dataset=small_train,
    processing_class=tokenizer,
    data_collator=data_collator,
)

diag_trainer.train()

Step,Training Loss
1,24.475626
2,30.075623
3,31.216005
4,21.012655
5,27.987787
6,21.377077
7,20.209984
8,20.021284
9,21.833632
10,21.032316


TrainOutput(global_step=13, training_loss=23.109708932729866, metrics={'train_runtime': 4.5958, 'train_samples_per_second': 10.879, 'train_steps_per_second': 2.829, 'total_flos': 6609371136000.0, 'train_loss': 23.109708932729866, 'epoch': 1.0})

## Diagnostic: Inspect a Raw Batch and Loss Directly
Debugging cell used to check whether the labels were being masked correctly
and whether the model produced a valid (non-NaN) loss on a real batch,
bypassing the Trainer entirely. This confirmed the labels were fine and
narrowed the problem down to the model's weights themselves being
corrupted leading to the fix of reloading a fresh model before the
successful training run.

In [ ]:
batch = data_collator([tokenized_datasets["train"][i] for i in range(4)])

print("Batch keys:", list(batch.keys()))
print("Labels shape:", batch["labels"].shape)
print("Sample label row (first 30 tokens):", batch["labels"][0][:30].tolist())
print("Non -100 label tokens in this batch:", (batch["labels"] != -100).sum().item())
print("Total label tokens in this batch:", batch["labels"].numel())

import torch
batch = {k: v.to("cuda") for k, v in batch.items()}
model.eval()
with torch.no_grad():
    out = model(**batch)
print("Direct forward-pass loss:", out.loss)

Batch keys: ['input_ids', 'attention_mask', 'labels', 'decoder_input_ids']
Labels shape: torch.Size([4, 128])
Sample label row (first 30 tokens): [431, 38956, 269, 2018, 14145, 259, 265, 265, 79592, 266, 19057, 576, 97507, 906, 259, 21804, 276, 259, 265, 10780, 11639, 321, 44025, 1708, 42857, 47246, 30384, 259, 265, 264]
Non -100 label tokens in this batch: 388
Total label tokens in this batch: 512
Direct forward-pass loss: tensor(13.1420, device='cuda:0')


## Diagnostic: Check Model Weights for Corruption
Scans every parameter tensor in the model for NaN or Inf values. This
confirmed that all 190 parameter tensors had become corrupted after an
earlier failed training attempt explaining why loss stayed frozen at
0.000000 across multiple runs regardless of the settings changed. This
check is also re-run after reloading a fresh model to confirm the new
copy is clean before training again.

In [ ]:
import torch

nan_params = 0
total_params = 0
for name, p in model.named_parameters():
    total_params += 1
    if torch.isnan(p).any() or torch.isinf(p).any():
        nan_params += 1
        print("Corrupted parameter:", name)

print(f"\n{nan_params} out of {total_params} parameter tensors contain NaN/Inf")


0 out of 190 parameter tensors contain NaN/Inf


## Step 12: Verify the Fresh Model Is Clean
Re-runs the corruption check on the newly reloaded model to confirm it has
zero NaN/Inf parameters before committing to a full training run. Confirmed
0 out of 190 corrupted the fresh model was safe to proceed with.

In [ ]:
nan_params = 0
for name, p in model.named_parameters():
    if torch.isnan(p).any() or torch.isinf(p).any():
        nan_params += 1
print(f"{nan_params} corrupted parameters (should be 0)")

0 corrupted parameters (should be 0)


## Step 13: Confirm the Fresh Model Trains Correctly
Re-runs the same 50-row quick training test as before, this time on the
freshly reloaded model. Real, non-zero, decreasing loss values (not
0.000000 or NaN) confirmed the fix worked and the model was ready for the
full 10-epoch training run.

In [ ]:
small_train = tokenized_datasets["train"].select(range(50))

diag_args = Seq2SeqTrainingArguments(
    output_dir=f'{PROJECT_DIR}/diag_checkpoints',
    eval_strategy="no",
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    learning_rate=3e-4,
    logging_strategy="steps",
    logging_steps=1,
    predict_with_generate=False,
    fp16=False,
    report_to="none",
)

diag_trainer = Seq2SeqTrainer(
    model=model,
    args=diag_args,
    train_dataset=small_train,
    processing_class=tokenizer,
    data_collator=data_collator,
)

diag_trainer.train()

Step,Training Loss
1,24.475626
2,30.075623
3,31.216005
4,21.012655
5,27.987787
6,21.377077
7,20.209984
8,20.021284
9,21.833632
10,21.032316


TrainOutput(global_step=13, training_loss=23.109708932729866, metrics={'train_runtime': 3.1155, 'train_samples_per_second': 16.049, 'train_steps_per_second': 4.173, 'total_flos': 6609371136000.0, 'train_loss': 23.109708932729866, 'epoch': 1.0})

## Step 14: Final Training Configuration
The corrected training configuration actually used for the real run
batch size lowered to 4 (fixes the GPU memory error), checkpoint retention
capped at 2 (fixes the Drive storage quota error), and fp16 disabled (part
of resolving the corrupted-weights issue). This is the version that
produced a working, successful training run.

In [10]:
training_args = Seq2SeqTrainingArguments(
    output_dir=f'{PROJECT_DIR}/checkpoints',
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=3e-4,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="sacrebleu",
    greater_is_better=True,
    save_total_limit=2,
    fp16=False,
    report_to="none",
)

## Step 15: Define Evaluation Metrics
Loads three translation-quality metrics — BLEU, sacreBLEU, and chrF — and
defines a function that decodes the model's predicted token IDs and true
labels back into readable text, then scores how closely they match. This
function runs automatically after every training epoch to track
translation quality over time.

Includes a fix sanitizing -100 padding tokens in the model's *predictions*
(not just the labels) — without this, evaluation crashed with an
OverflowError whenever the model generated a shorter sequence than the
maximum length.

In [11]:
sacrebleu_metric = evaluate.load("sacrebleu")
bleu_metric = evaluate.load("bleu")
chrf_metric = evaluate.load("chrf")

def postprocess_text(preds, labels):
    preds = [p.strip() if p.strip() != "" else "empty" for p in preds]
    labels = [[l.strip()] for l in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Sanitize predictions: replace any -100 padding before decoding
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels_sb = postprocess_text(decoded_preds, decoded_labels)

    sacrebleu_result = sacrebleu_metric.compute(predictions=decoded_preds, references=decoded_labels_sb)
    chrf_result = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels_sb)

    try:
        bleu_result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels_sb)
        bleu_score = round(bleu_result["bleu"] * 100, 2)
    except ZeroDivisionError:
        bleu_score = 0.0

    return {
        "bleu": bleu_score,
        "sacrebleu": round(sacrebleu_result["score"], 2),
        "chrf": round(chrf_result["score"], 2),
    }

## Step 16: Train the Model
Builds the Trainer with the corrected configuration and starts the actual
10-epoch fine-tuning run. After training completes, saves the full
per-epoch training log to Drive for use in the final report, and displays
the last 20 logged rows for a quick check.

In [12]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train(resume_from_checkpoint=f'{PROJECT_DIR}/checkpoints/checkpoint-19236')

log_history = pd.DataFrame(trainer.state.log_history)
log_history.to_csv(f'{PROJECT_DIR}/training_log.csv', index=False)
log_history.tail(20)

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


RuntimeError: params, grads, exp_avgs, and exp_avg_sqs must have same dtype, device, and layout

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(f'{PROJECT_DIR}/checkpoints/checkpoint-19236')
model = model.to("cuda")

nan_params = sum(1 for _, p in model.named_parameters() if torch.isnan(p).any() or torch.isinf(p).any())
print(f"Corrupted parameters: {nan_params} (should be 0)")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Corrupted parameters: 0 (should be 0)


## Step 17: Save and Package the Final Model
Saves the best-performing checkpoint (automatically restored by
load_best_model_at_end) along with its tokenizer to Drive, prints which
epoch was selected as best and its sacreBLEU score, then zips the model
folder into a single file ready for submission.

In [14]:
trainer.save_model(f'{PROJECT_DIR}/final_model')
tokenizer.save_pretrained(f'{PROJECT_DIR}/final_model')

print("Best checkpoint metric (sacrebleu):", trainer.state.best_metric)
print("Best checkpoint path:", trainer.state.best_model_checkpoint)

!cd {PROJECT_DIR} && zip -r final_model.zip final_model
print("Zipped model saved to:", f'{PROJECT_DIR}/final_model.zip')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best checkpoint metric (sacrebleu): 69.87
Best checkpoint path: /content/drive/MyDrive/NLP_Group_Project/Patricia_mT5_Somali/checkpoints/checkpoint-19236
  adding: final_model/ (stored 0%)
  adding: final_model/config.json (deflated 49%)
  adding: final_model/generation_config.json (deflated 59%)
  adding: final_model/model.safetensors (deflated 25%)
  adding: final_model/tokenizer_config.json (deflated 84%)
  adding: final_model/tokenizer.json (deflated 76%)
  adding: final_model/training_args.bin (deflated 53%)
Zipped model saved to: /content/drive/MyDrive/NLP_Group_Project/Patricia_mT5_Somali/final_model.zip


## Step 18: Generate Sample Translations
Picks 8 random sentences from the held-out test set and runs them through
the fine-tuned model to see real English→Somali translations, comparing
each prediction against its reference translation. Saved to Drive for use
in the report's Sample Translations section.

In [15]:
sample_test = test_df.sample(n=8, random_state=42).reset_index(drop=True)

def translate(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to("cuda")
    outputs = model.generate(**inputs, max_length=MAX_TARGET_LEN)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

sample_test["prediction"] = sample_test["source"].apply(translate)
sample_test[["source", "target", "prediction"]].to_csv(f'{PROJECT_DIR}/sample_translations.csv', index=False)
sample_test[["source", "target", "prediction"]]

,source,target,prediction
0,Local leaders are working with residents on me...,Hogaamiyayaasha deegaanku waxay kala shaqaynay...,Hogaamiyayaasha deegaanku waxay dadka deegaank...
1,Community members can access applying for offi...,Xubnaha bulshadu waxay ka heli karaan codsiga ...,Xubnaha bulshadu waxay heli karaan codsiga duk...
2,This month's public update covers farmer regis...,Cusboonaysiinta dadwaynaha ee bishan waxa ay q...,Cusboonaysiinta dadwaynaha ee bishan waxa ay k...
3,Access early warning information on flooding f...,Hel macluumaadka digniinta hore ee fatahaadaha...,Helitaanka macluumaadka hore ee ku saabsan fat...
4,This month's public update covers Huduma Namba...,Cusboonaysiinta dadwaynaha ee bishan waxa ay k...,Cusboonaysiinta dadwaynaha ee bishan waxa ay q...
5,Local offices are coordinating efforts on acce...,Xafiisyada maxalliga ah ayaa isku dubaridinaya...,Xafiisyada maxalliga ah ayaa isku dubaridinaya...
6,Households are reminded of three ongoing progr...,Qoysaska waxa la xasuusinayaa saddex barnaamij...,Qoysaska waxa la xasuusinayaa saddex barnaamij...
7,Stay informed on the latest details of the Nat...,La soco tafaasiisha ugu danbeysa ee barnaamijk...,La soco tafaasiisha ugu danbeysa ee barnaamijk...


Untruncated content

In [16]:
pd.set_option('display.max_colwidth', None)
for i, row in sample_test.iterrows():
    print(f"--- Example {i+1} ---")
    print("Source:", row['source'])
    print("Target:", row['target'])
    print("Prediction:", row['prediction'])
    print()

--- Example 1 ---
Source: Local leaders are working with residents on mental health support services under the national mental health action plan.
Target: Hogaamiyayaasha deegaanku waxay kala shaqaynayaan dadka deegaanka adeegyada taageerada caafimaadka dhimirka ee hoos yimaada qorshaha waxqabadka caafimaadka dhimirka ee qaranka.
Prediction: Hogaamiyayaasha deegaanku waxay dadka deegaanka kala shaqaynayaan adeegyada taageerada caafimaadka dhimirka ee hoos yimaada qorshaha waxqabadka caafimaadka dhimirka ee qaranka.

--- Example 2 ---
Source: Community members can access applying for official documents through the eCitizen online platform as well as filing police oversight complaints with the Independent Policing Oversight Authority (IPOA) through local offices.
Target: Xubnaha bulshadu waxay ka heli karaan codsiga dukumeentiyada rasmiga ah ee eCitizen onlaynka ah iyo sidoo kale u gudbinta cabashooyinka kormeerka bilayska Maamulka Kormeerka Bilayska ee Madaxbanaan (IPOA) iyada oo loo ma

In [4]:
import os
print("checkpoint-19236:", os.listdir(f'{PROJECT_DIR}/checkpoints/checkpoint-19236'))
print("checkpoint-16030:", os.listdir(f'{PROJECT_DIR}/checkpoints/checkpoint-16030'))

checkpoint-19236: ['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth', 'trainer_state.json']
checkpoint-16030: ['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth', 'trainer_state.json']
